# Phase 4 — Train model_25m on Kaggle TPU v5e-8

**Before you start:**
1. Mount the `llm-forge-tokens-v1` dataset from the Kaggle sidebar (+ Add data → search `adeshboudh/llm-forge-tokens-v1`).
2. Run the cells in order. Cells 1-3 install the repo + jax/jaxlib/libtpu; cells 4-6 verify the env; cell 7+ run training.

**Why a separate jax/jaxlib/libtpu install?** `uv sync` in cell 2 installs jax from pypi per pyproject's `jax>=0.4.20` pin, which on Kaggle's Python 3.14 returns the CPU jaxlib. Cell 3 forces a reinstall of jax/jaxlib/libtpu from pypi into the venv. The pypi jaxlib 0.10.2 is TPU-capable (jaxlib is built for all backends), and the libtpu pip package finds the pre-installed libtpu.so on the Kaggle TPU VM. After cell 3, `jax.devices()` returns 8 TPU devices.

In [ ]:
!git clone https://github.com/adeshboudh/llm_forge.git 2>/dev/null || (cd llm_forge && git pull)

In [ ]:
!pip install uv && cd llm_forge && uv sync --extra dev

In [ ]:
# Reinstall jax/jaxlib/libtpu into the venv from pypi (NOT the libtpu_releases
# index, which only has pre-release nightly libtpu wheels). jax 0.10.2 + jaxlib
# 0.10.2 from pypi has cp314 wheels and is TPU-capable; the libtpu pip package
# finds the pre-installed libtpu.so on the Kaggle TPU VM (no compile needed).
# --reinstall overrides whatever uv sync installed in cell 2.
!cd llm_forge && uv pip install --reinstall jax jaxlib libtpu

In [ ]:
!cd llm_forge && uv run python -c "import jax; print('jax:', jax.__version__); print('backend:', jax.default_backend()); print('devices:', jax.devices())"

In [ ]:
!ls /kaggle/input/datasets/adeshboudh/llm-forge-tokens-v1/ | head -5 && echo "---" && ls /kaggle/input/datasets/adeshboudh/llm-forge-tokens-v1/ | wc -l && echo "shards above (expect 216: 215 npy + metadata.json)"

In [ ]:
!cd llm_forge && uv run python -m training.summary --config configs/training/model_25m.yaml

In [ ]:
# Sanity: 50 steps with small batch + seq_len to avoid TPU HBM OOM.
# (The full model_25m.yaml config uses batch=128, seq=1024 which overflows
#  single-core HBM during the attention score matmul. Sanity uses batch=8,
#  seq=256 to keep the score tensor tiny while still exercising the train loop.)
!cd llm_forge && uv run python -m training.train --config configs/training/model_25m.yaml --max-steps 50 --batch-size 8 --seq-len 256

In [ ]:
# Full 1B-token run (9766 steps, ~3-6 hours on 8 v5e cores).
!cd llm_forge && uv run python -m training.train --config configs/training/model_25m.yaml

In [ ]:
import json
import matplotlib.pyplot as plt

rows = [json.loads(l) for l in open('/kaggle/working/train_log.jsonl')]
steps = [r['step'] for r in rows]
losses = [r['loss'] for r in rows]
val = [(r['step'], r['val_loss']) for r in rows if r.get('val_loss') is not None]

plt.plot(steps, losses, label='train')
if val:
    plt.plot(*zip(*val), 'o-', label='val')
plt.xlabel('step'); plt.ylabel('loss'); plt.legend(); plt.show()

In [ ]:
# Load final checkpoint and generate 3 samples (smoke; real sampling in Phase 6).
import os, sys
REPO = '/kaggle/working/llm_forge'
sys.path.insert(0, REPO)  # notebook kernel is python3.12; repo lives outside its site-packages
os.chdir(REPO)           # so relative paths (configs/...) resolve
import jax, jax.numpy as jnp
from pathlib import Path
from model.config import load_model_config
from model.lm import LM
from training.state import create_train_state, restore
from training.config import load_training_config

print(f'CWD: {os.getcwd()}')
print(f'configs/training/ exists: {Path("configs/training").exists()}')
if Path('configs/training').exists():
    print(f'  contents: {sorted(p.name for p in Path("configs/training").iterdir())}')

cfg = load_model_config('model_25m')
train_cfg = load_training_config('configs/training/model_25m.yaml')
model = LM(config=cfg)
state = create_train_state(jax.random.PRNGKey(0), model, train_cfg, cfg)  # placeholder params
# state = restore('/kaggle/working/ckpt/step_000009766_final', state)  # uncomment after full run

prompt = jnp.array([[3, 4, 5, 6]], dtype=jnp.int32)  # <|bos|> + 3 random tokens
loss, logits = model.apply(state.params, prompt, prompt, return_logits=True)
print('logits shape:', logits.shape, '(full sampling lives in Phase 6)')
next_token = int(jnp.argmax(logits[0, -1]))
print(f'argmax of last-position logits -> token id {next_token} (vocab 32768; untrained -> mostly noise)')